In [1]:
import os
import whisper
import re
import nltk
import string
import numpy as np
import tensorflow_hub as hub
from transformers import pipeline
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from joblib import load


nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mahmo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mahmo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mahmo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mahmo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mahmo\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
# Load the model (choose from 'tiny', 'base', 'small', 'medium', 'large')
whisper_model = whisper.load_model("small")

C:\Users\mahmo\AppData\Local\Programs\Python\Python311\Lib\site-packages\whisper\__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, m

In [3]:
filepath = input("Please enter the path of the record file you want to preprocess: ")

data = whisper_model.transcribe(filepath)

texts = [data['text']]

print(texts)

C:\Users\mahmo\AppData\Local\Programs\Python\Python311\Lib\site-packages\whisper\transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


[" In December 2019, there was a cluster of pneumonia cases in the city of Wuhan in China. Some of the early cases had reported visiting or working in a seafood and live animal market in Wuhan. Investigations found that the disease was caused by a newly discovered coronavirus. The disease was subsequently named COVID-19. COVID-19 spread within China and to the rest of the world. On 30 January 2020, the World Health Organization declared the outbreak a public health emergency of international concern. In this video, we'll take a quick look at what is currently known about COVID-19. Keep in mind that this is a new disease and what's known is rapidly evolving and might change in the future. So what is a coronavirus? Coronaviruses are a large group of viruses. They consist of a core of genetic material surrounded by a lipid envelope with protein spikes. This gives it the appearance of a crown. Crown in Latin is called corona, and that's how these viruses get their name. There are different

In [5]:
# Combine all preprocessing steps
def preprocess_text(text):

    def to_lower(text):
        return text.lower()

    def remove_punctuation_numbers(text):
        text = re.sub(r'\d+', '', text)  # Remove numbers
        text = re.sub(r'\b(\d+)(st|nd|rd|th)\b', '', text)  # Remove Ordered Numbering
        return text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation

    def tokenize(text):
        return nltk.word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    def remove_stopwords(tokens):
        return [word for word in tokens if word not in stop_words]

    lemmatizer = WordNetLemmatizer()
    def apply_lemmatization(tokens):
        return [lemmatizer.lemmatize(word) for word in tokens]

    text = to_lower(text)
    text = remove_punctuation_numbers(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = apply_lemmatization(tokens)
    return " ".join(tokens)


processed_input = preprocess_text(texts[0])

In [15]:
# Initialize the abstractive summarization pipeline
abstractive_summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Initialize the extractive summarizer
summarizer = LsaSummarizer()

# Function for abstractive summarization
def abstractive_summary(text):
    try:
        summary = abstractive_summarizer(text, max_length=150, min_length=30, do_sample=False)[0]['summary_text']
        return summary
    except Exception as e:
        return f"Error in abstractive summarization: {str(e)}"

# Function for extractive summarization
def extractive_summary(text):
    try:
        parser = PlaintextParser.from_string(text, Tokenizer("english"))
        summary = summarizer(parser.document, 2)
        return " ".join([str(sentence) for sentence in summary])
    except Exception as e:
        return f"Error in extractive summarization: {str(e)}"

print(f'Abstractive Summary of input: {abstractive_summary(processed_input)}')
print(f'Extractive Summary of input: {extractive_summary(processed_input)}')

Abstractive Summary of input:  coronavirus disease subsequently named covid covid spread within china rest world january world health organization declared outbreak public health emergency international concern video well take quick look currently known covid keep mind new disease whats known rapidly evolving might change future coronav virus causes illness animal human human coronaviruses cause respiratory infection ranging common cold severe disease.
Extractive Summary of input: december cluster pneumonia case city wuhan china early case reported visiting working seafood live animal market wuhan investigation found disease caused newly discovered coronavirus disease subsequently named covid covid spread within china rest world january world health organization declared outbreak public health emergency international concern video well take quick look currently known covid keep mind new disease whats known rapidly evolving might change future coronavirus coronaviruses large group virus

In [12]:
# Predict the sentiment of the input
from tensorflow.keras.models import load_model

model = load_model('Deployment/model.keras')
encoder = load('Deployment/label_encoder.joblib')

embed = hub.load("https://tfhub.dev/google/nnlm-en-dim128/2")

embed = embed([processed_input])

model.predict(embed)

print(f'Predicted Topic: {encoder.inverse_transform(np.argmax(model.predict(embed), axis=1))}')

C:\Users\mahmo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 12 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predicted Topic: ['Public Awareness Measures']
